# 🧹 Task 2 — Data Cleaning & Preprocessing
**DecodeLabs Data Analytics Internship | Batch 2026**  
**Intern:** Umm-e-Abiha

---

## 🎯 Objective
Transform the raw dataset into a clean, analysis-ready format by addressing data quality issues systematically.

## 📋 Deliverables
- Detect and handle all missing values
- Identify and remove duplicate records
- Parse and enrich date columns
- Validate computed fields (TotalPrice)
- Standardise categorical text values
- Produce a professional Change Log

## 🧠 Skills Applied
`data quality` · `pandas` · `missing value imputation` · `documentation`

---

## ⚙️ Environment Setup & Data Load

In [ ]:
# ── Core Libraries ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Machine Learning ─────────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

# ── Global Plot Style ────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d0d0d',
    'axes.facecolor'  : '#1a1a2e',
    'axes.edgecolor'  : '#00d4ff',
    'axes.labelcolor' : '#ffffff',
    'xtick.color'     : '#cccccc',
    'ytick.color'     : '#cccccc',
    'text.color'      : '#ffffff',
    'grid.color'      : '#2a2a4a',
    'grid.linestyle'  : '--',
    'grid.alpha'      : 0.4,
})

CYAN   = '#00d4ff'
ORANGE = '#ff6b35'
GREEN  = '#39ff14'
PURPLE = '#bf5af2'
YELLOW = '#ffd60a'
PINK   = '#ff2d78'
COLORS = [CYAN, ORANGE, GREEN, PURPLE, YELLOW, PINK, '#00ff88']

print("✅ All libraries imported successfully!")

df_raw = pd.read_excel('Dataset_for_Data_Analytics.xlsx')
df = df_raw.copy()   # Preserve original dataset
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

## 2.1 Step 1 — Missing Value Audit

In [ ]:
# ── Missing Value Detection ──────────────────────────────────────────────────
print(f"{'Column':<22} {'Missing':>8} {'%':>8}  {'Status'}")
print("─" * 52)
missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
for col in df.columns:
    status = "⚠️  ACTION REQUIRED" if missing[col] > 0 else "✅ Complete"
    print(f"  {col:<20} {missing[col]:>8}  {missing_pct[col]:>6}%  {status}")


## 2.2 Step 2 — Handle Missing Values

In [ ]:
# ── Impute CouponCode Missing Values ─────────────────────────────────────────
# Rationale: A missing CouponCode means the customer did not apply a coupon.
# We impute with 'NO_COUPON' to preserve the 309 records (not delete them).

before = df['CouponCode'].isnull().sum()
df['CouponCode'] = df['CouponCode'].fillna('NO_COUPON')
after  = df['CouponCode'].isnull().sum()

print("CouponCode — Missing Value Treatment:")
print(f"   Before imputation : {before} missing")
print(f"   After imputation  : {after} missing")
print(f"   Records preserved : {before} (imputed with 'NO_COUPON')")
print("   Rationale: missing = coupon was not used, not a data error")


## 2.3 Step 3 — Duplicate Record Check

In [ ]:
# ── Duplicate Detection & Removal ────────────────────────────────────────────
dupes_before = df.duplicated().sum()
df = df.drop_duplicates()
dupes_after  = df.duplicated().sum()

print("Duplicate Record Audit:")
print(f"   Duplicates found   : {dupes_before}")
print(f"   Duplicates removed : {dupes_before - dupes_after}")
print(f"   Remaining rows     : {len(df):,}")
print(f"   Dataset integrity  : {'✅ No duplicates' if dupes_after == 0 else '⚠️ Check required'}")


## 2.4 Step 4 — Date Parsing & Feature Extraction

In [ ]:
# ── Date Column Conversion ───────────────────────────────────────────────────
print(f"Date dtype before : {df['Date'].dtype}")

df['Date']      = pd.to_datetime(df['Date'])
df['Year']      = df['Date'].dt.year
df['Month']     = df['Date'].dt.month
df['MonthName'] = df['Date'].dt.strftime('%b')

print(f"Date dtype after  : {df['Date'].dtype}")
print(f"Date range        : {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"New features added: Year, Month, MonthName")
print()
print(df[['Date', 'Year', 'Month', 'MonthName']].head(4).to_string(index=False))


## 2.5 Step 5 — TotalPrice Validation

In [ ]:
# ── Validate TotalPrice = Quantity × UnitPrice ───────────────────────────────
df['_Calc_Total']   = (df['Quantity'] * df['UnitPrice']).round(2)
df['_TotalPrice_r'] = df['TotalPrice'].round(2)
mismatches = (df['_Calc_Total'] != df['_TotalPrice_r']).sum()

print("TotalPrice Integrity Check:")
print(f"   Formula applied   : Quantity × UnitPrice")
print(f"   Mismatches found  : {mismatches}")
print(f"   Verdict           : {'✅ All values correct' if mismatches == 0 else '⚠️ ' + str(mismatches) + ' errors found'}")

df.drop(columns=['_Calc_Total', '_TotalPrice_r'], inplace=True)


## 2.6 Step 6 — Text Standardisation

In [ ]:
# ── Standardise Categorical Text (strip + title case) ────────────────────────
text_cols = ['Product', 'OrderStatus', 'PaymentMethod', 'ReferralSource']
for col in text_cols:
    df[col] = df[col].str.strip().str.title()

print("Text columns standardised (whitespace stripped + Title Case applied):")
for col in text_cols:
    print(f"   {col:<20}: {df[col].unique().tolist()}")


## 2.7 Cleaned Dataset Summary

In [ ]:
# ── Final Quality Report ─────────────────────────────────────────────────────
print("=" * 50)
print("   CLEANED DATASET — QUALITY REPORT")
print("=" * 50)
print(f"   Rows          : {len(df):,}")
print(f"   Columns       : {len(df.columns)}")
print(f"   Missing values: {df.isnull().sum().sum()}")
print(f"   Duplicates    : {df.duplicated().sum()}")
print(f"   Date range    : {df['Date'].min().date()} → {df['Date'].max().date()}")
print("   Status        : ✅ Dataset is clean and analysis-ready")
print("=" * 50)


## 2.8 Professional Change Log

In [ ]:
# ── Change Log (Professional Documentation Standard) ─────────────────────────
change_log = pd.DataFrame({
    'Change_ID'  : ['CR-001', 'CR-002', 'CR-003', 'CR-004', 'CR-005'],
    'Column'     : ['CouponCode', 'All columns', 'Date', 'TotalPrice', 'Categorical'],
    'Action'     : [
        'Filled 309 NaN with NO_COUPON',
        'Removed 0 duplicate rows',
        'Parsed to datetime; extracted Year, Month, MonthName',
        'Validated: Qty × UnitPrice = TotalPrice',
        'Applied str.strip() + str.title()'
    ],
    'Impact'     : [
        '309 records preserved',
        'No data lost',
        'Time-series analysis enabled',
        '0 mismatches confirmed',
        'Consistent category labels'
    ],
    'Status'     : ['✅ Resolved'] * 5
})
print("Change Log:")
change_log


## 2.9 Visualisation — Cleaning Impact

In [ ]:
# ── Task 2 Visualisation ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle('Task 2 — Data Cleaning Summary', color=CYAN, fontsize=14, fontweight='bold')

# (a) Missing Values — Before vs After
cats   = ['CouponCode\nBefore', 'CouponCode\nAfter', 'Other Columns']
vals   = [309, 0, 0]
bcolors = [ORANGE, GREEN, CYAN]
bars = axes[0].bar(cats, vals, color=bcolors, edgecolor='#0d0d0d', linewidth=1.5, width=0.5)
axes[0].set_title('Missing Values: Before vs After Cleaning', color=CYAN, fontsize=12)
axes[0].set_ylabel('Missing Count', color='white')
for bar, v in zip(bars, vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 5,
                 str(v), ha='center', color='white', fontweight='bold', fontsize=13)

# (b) Payment Method Distribution (clean data)
pay_counts = df['PaymentMethod'].value_counts()
axes[1].bar(pay_counts.index, pay_counts.values,
            color=COLORS[:len(pay_counts)], edgecolor='#0d0d0d', linewidth=1.5)
axes[1].set_title('Payment Method Distribution (Clean Data)', color=CYAN, fontsize=12)
axes[1].set_ylabel('Order Count', color='white')
axes[1].tick_params(axis='x', rotation=20)
for i, v in enumerate(pay_counts.values):
    axes[1].text(i, v + 2, str(v), ha='center', color='white', fontsize=10)

plt.tight_layout()
plt.savefig('task2_cleaning.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print("✅ Chart saved → task2_cleaning.png")


## 💡 Task 2 — Key Findings

| Step | Action Taken | Business Rationale |
|------|-------------|--------------------|
| CR-001 | 309 missing `CouponCode` → `NO_COUPON` | Imputation preserves records; absence of coupon is valid data |
| CR-002 | Duplicate check — 0 found | Dataset has no redundant entries |
| CR-003 | `Date` → `datetime`; extracted `Year`, `Month`, `MonthName` | Enables time-series analysis |
| CR-004 | `TotalPrice` validation — 0 mismatches | Confirms computational accuracy |
| CR-005 | Text standardised — strip + title case | Prevents groupby fragmentation from whitespace/case issues |

> **Industry Principle Applied:** *"80% of a Data Scientist's work is data preparation; only 20% is analysis."*  
> A well-documented change log ensures reproducibility and audit-readiness.
